# 08 - Logistic Regression and Final Site Selection

This notebook continues directly from notebook 07 and answers two questions:

**Part A - Which of the top 4 postcodes is the best place to open a new Migros?**
Notebook 07 ranked candidates with a hand-weighted opportunity score (40% population,
25% income, 25% distance, 10% low competition). Those weights were chosen by me, not by
the data. Here I replace them with a **logistic regression** that learns from the ~3,170
Swiss postcodes what a "Migros postcode" actually looks like, and then asks the model:
*given its profile, how likely is it that this postcode should have a Migros?*
A postcode with a high predicted probability but **zero** actual Migros stores is a real
coverage gap, not just a high score.

**Part B - Where exactly inside Siebnen should the store go?**
Postcode-level analysis stops at "8854". To pinpoint a site inside the village I build a
100 m grid over Siebnen and score every grid cell with a **Huff gravity model**
(expected customers captured from surrounding demand, given the competition already
present), combined with an accessibility filter along the main retail road. The result is
a map with three concrete site suggestions and their coordinates.

## 0. Setup

In [ ]:
# Core data-handling libraries (same stack as notebooks 01-07 so nothing new is introduced)
import pandas as pd                      # tables / dataframes
import numpy as np                       # vectorised maths, used heavily for distance calculations
import zipfile                           # to read the official locality file without unpacking it manually
from pathlib import Path                 # OS-independent file paths

# Plotting (Plotly, to stay consistent with the interactive charts of notebook 07)
import plotly.express as px
import plotly.graph_objects as go

# Modelling
import statsmodels.api as sm                                  # Logit with a full statistical summary (p-values, CI)
from sklearn.model_selection import train_test_split          # honest hold-out evaluation
from sklearn.metrics import roc_auc_score, confusion_matrix, accuracy_score
from sklearn.neighbors import BallTree                        # fast nearest-neighbour search on a sphere

# Show every column when printing wide dataframes, otherwise pandas truncates the output
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

# Folder where the interactive HTML figures of this notebook will be written at the end
plot_folder = Path("plots")
plot_folder.mkdir(parents=True, exist_ok=True)

# Folder holding the cleaned datasets produced by notebooks 01-06
data_folder = Path("data")

EARTH_RADIUS_KM = 6371   # mean Earth radius; used to convert angular distances into kilometres

In [ ]:
# Plotly renamed its map traces in version 5.24 (MapLibre replaced Mapbox).
# This small check keeps the notebook working on both old and new Plotly versions.
import plotly

plotly_version = tuple(int(part) for part in plotly.__version__.split(".")[:2])
USE_NEW_MAPS = plotly_version >= (5, 24)      # True -> Scattermap/Densitymap, False -> Scattermapbox/Densitymapbox

# Pick the right trace classes once, so the plotting code further down stays readable
ScatterMapTrace = go.Scattermap if USE_NEW_MAPS else go.Scattermapbox
DensityMapTrace = go.Densitymap if USE_NEW_MAPS else go.Densitymapbox

def apply_map_layout(figure, center_lat, center_lon, zoom):
    """Set the basemap style/centre/zoom using whichever API this Plotly version supports."""
    # OpenStreetMap needs no API token, which is why it is used here
    settings = dict(style="open-street-map", center=dict(lat=center_lat, lon=center_lon), zoom=zoom)
    if USE_NEW_MAPS:
        figure.update_layout(map=settings)     # new MapLibre-based property
    else:
        figure.update_layout(mapbox=settings)  # legacy Mapbox property
    return figure

print("Plotly version:", plotly.__version__, "| using new map traces:", USE_NEW_MAPS)

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km between two points (or arrays of points) given in degrees.

    Used everywhere in this notebook because postcode centres and stores are given as
    latitude/longitude, and straight-line degree differences are not distances.
    """
    # Trigonometric functions in numpy expect radians, not degrees
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    # Standard haversine formula
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2

    # arcsin form is numerically stable for the small distances used here
    return 2 * EARTH_RADIUS_KM * np.arcsin(np.sqrt(a))

In [ ]:
# Load the master table built in notebook 06 (one row per postcode)
# postal_code is read as a string so that leading zeros are never lost
analysis_df = pd.read_csv(data_folder / "migros_master_data.csv", dtype={"postal_code": "string"})

# Keep only postcodes with complete income, municipality and coordinate information,
# exactly as in notebook 07, so both notebooks work on the identical population of rows
scoring_df = analysis_df[analysis_df["complete_location_data"] == True].copy().reset_index(drop=True)

print("All postcodes:", analysis_df.shape)
print("Postcodes used for modelling:", scoring_df.shape)
scoring_df.head(3)

## 1. Rebuild the notebook 07 ranking and take the top 4

The logistic regression has to be applied to the *same* candidates notebook 07 produced, so
the ranking is recomputed here instead of being copied by hand. This also keeps the notebook
reproducible if the underlying data is ever refreshed.

In [ ]:
# Candidate postcodes are, as in notebook 07, postcodes in the three screened cantons
# (Zurich, Schwyz, Aargau) that currently have no Migros supermarket at all
selected_cantons = ["ZH", "SZ", "AG"]

opportunity_df = scoring_df[
    scoring_df["primary_canton"].isin(selected_cantons) &     # only the three cantons flagged by the canton-level screening
    (scoring_df["migros_count"] == 0)                         # only genuine white spots
].copy()

# Postcodes that already have at least one Migros - these are the reference points for
# "how far is the nearest existing Migros?"
migros_postcodes_df = scoring_df[scoring_df["migros_count"] > 0].copy()

# A BallTree with the haversine metric searches nearest neighbours on the sphere in O(log n),
# which is much faster than comparing every candidate with every Migros postcode
migros_tree = BallTree(np.radians(migros_postcodes_df[["postcode_latitude", "postcode_longitude"]]), metric="haversine")

# Query the single nearest Migros postcode for every candidate
nearest_distance, nearest_index = migros_tree.query(np.radians(opportunity_df[["postcode_latitude", "postcode_longitude"]]), k=1)

# BallTree returns angular distance in radians -> multiply by the Earth radius to get kilometres
opportunity_df["nearest_migros_distance_km"] = nearest_distance.flatten() * EARTH_RADIUS_KM

# Keep the postcode of that nearest Migros for the summary tables
opportunity_df["nearest_migros_postcode"] = migros_postcodes_df.iloc[nearest_index.flatten()]["postal_code"].to_numpy()

In [ ]:
# Reproduce the weighted opportunity score of notebook 07
from sklearn.preprocessing import MinMaxScaler

# Min-max scaling puts population, income, distance and competitor counts on a common 0-1 range
# so that they can be added up despite having completely different units
scaler = MinMaxScaler()
score_inputs = ["population", "avg_income_per_taxpayer", "nearest_migros_distance_km", "external_competitor_count"]
opportunity_df[["population_score", "income_score", "distance_score", "competitor_level"]] = scaler.fit_transform(opportunity_df[score_inputs])

# For competitors a low value was treated as desirable, so the scaled value is inverted
opportunity_df["low_competition_score"] = 1 - opportunity_df["competitor_level"]

# The hand-chosen weights from notebook 07 (this is exactly the subjectivity the model will replace)
opportunity_df["opportunity_score"] = 100 * (
    opportunity_df["population_score"] * 0.40 +
    opportunity_df["income_score"] * 0.25 +
    opportunity_df["distance_score"] * 0.25 +
    opportunity_df["low_competition_score"] * 0.10
)

# Rank the candidates and keep the four best ones - these are the postcodes this notebook evaluates
ranking_df = opportunity_df.sort_values("opportunity_score", ascending=False).reset_index(drop=True)
ranking_df["rank"] = ranking_df.index + 1

top4_df = ranking_df.head(4).copy()
top4_postcodes = top4_df["postal_code"].tolist()

display(
    top4_df[["rank", "postal_code", "locality_name", "municipality_name", "primary_canton",
             "population", "avg_income_per_taxpayer", "external_competitor_count",
             "nearest_migros_distance_km", "opportunity_score"]].style.format({
        "population": "{:,.0f}",
        "avg_income_per_taxpayer": "CHF {:,.0f}",
        "nearest_migros_distance_km": "{:.2f} km",
        "opportunity_score": "{:.1f}"
    })
)

## 2. Part A - Logistic regression

### 2.1 Why logistic regression here?

The question "should there be a Migros in this postcode?" has a **binary** answer, so linear
regression is the wrong tool (it would predict values below 0 and above 1). Logistic
regression models the *probability* of the binary outcome and gives interpretable odds ratios.

- **Target (y):** `has_migros` - does the postcode contain at least one Migros supermarket? (1/0)
- **Training rows:** all 3,170 postcodes with complete data, *not* only the candidates. The model
  needs to see both covered and uncovered postcodes to learn the difference.
- **Features (X):**

| Feature | Why it is included |
|---|---|
| `log10(population)` | Store viability scales with local customers; logs are used because population is heavily right-skewed |
| `avg_income_per_taxpayer / 10,000` | Purchasing power; divided by 10k so the coefficient is per CHF 10,000 instead of per franc |
| `external_competitor_count` | Coop/Aldi/Lidl presence is evidence that the location is commercially attractive |
| `log10(catchment population within 5 km)` | A postcode is not an island - surrounding demand matters, and it also captures urban vs rural context |

**Deliberately excluded:** distance to the nearest Migros. For every postcode that *has* a Migros
that distance is 0, so the feature would leak the answer and the model would learn nothing.

In [ ]:
# Feature 4 requires the population living within 5 km of each postcode centre.
# Build one BallTree over ALL scoring postcodes and query a radius instead of k neighbours.
all_coordinates_rad = np.radians(scoring_df[["postcode_latitude", "postcode_longitude"]].to_numpy())
all_postcode_tree = BallTree(all_coordinates_rad, metric="haversine")

# query_radius returns, for every postcode, the indices of all postcodes within 5 km
# (radius must be given in radians: km / Earth radius)
neighbour_indices = all_postcode_tree.query_radius(all_coordinates_rad, r=5 / EARTH_RADIUS_KM)

# Sum the population of those neighbours, then subtract the postcode's own population
# so the feature describes the SURROUNDING area rather than repeating feature 1
population_values = scoring_df["population"].to_numpy()
scoring_df["catchment_population_5km"] = [population_values[idx].sum() for idx in neighbour_indices] - scoring_df["population"]

# Log transforms: both population variables span several orders of magnitude (a few hundred to >100,000).
# +1 avoids log(0) for empty surroundings.
scoring_df["log_population"] = np.log10(scoring_df["population"] + 1)
scoring_df["log_catchment_population"] = np.log10(scoring_df["catchment_population_5km"] + 1)

# Rescale income so one unit = CHF 10,000, which makes the odds ratio readable
scoring_df["income_10k"] = scoring_df["avg_income_per_taxpayer"] / 10000

# Target variable as integer 0/1 (statsmodels expects numeric, not boolean)
scoring_df["has_migros_flag"] = scoring_df["has_migros"].astype(int)

# Quick sanity check on class balance - logistic regression is fine with imbalance,
# but the baseline accuracy has to be known before judging the model
print(scoring_df["has_migros_flag"].value_counts())
print("Share of postcodes with a Migros:", f"{scoring_df['has_migros_flag'].mean():.1%}")

In [ ]:
# Assemble the design matrix
feature_names = ["log_population", "income_10k", "external_competitor_count", "log_catchment_population"]

X = scoring_df[feature_names]          # predictors
y = scoring_df["has_migros_flag"]      # target

# Hold out 30% of the postcodes so model quality is measured on data the model never saw.
# stratify=y keeps the same share of Migros postcodes in both parts.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)

# statsmodels does NOT add an intercept automatically, so it is added explicitly
X_train_const = sm.add_constant(X_train)
X_test_const = sm.add_constant(X_test)

# Fit the logistic regression by maximum likelihood (disp=0 hides the iteration log)
logit_train_model = sm.Logit(y_train, X_train_const).fit(disp=0)

# Predicted probabilities on the unseen test set
test_probabilities = logit_train_model.predict(X_test_const)

# Convert probabilities to a 0/1 decision at the usual 0.5 threshold
test_predictions = (test_probabilities >= 0.5).astype(int)

print("Test accuracy:", f"{accuracy_score(y_test, test_predictions):.3f}")
print("Test ROC AUC :", f"{roc_auc_score(y_test, test_probabilities):.3f}")   # AUC is threshold-independent
print()
print("Confusion matrix (rows = actual, columns = predicted):")
print(pd.DataFrame(confusion_matrix(y_test, test_predictions),
                   index=["actual: no Migros", "actual: Migros"],
                   columns=["predicted: no Migros", "predicted: Migros"]))

In [ ]:
# The hold-out test confirmed the model generalises, so it is now refitted on ALL postcodes:
# more data gives more stable coefficients, and every candidate is then scored by the same model.
X_full_const = sm.add_constant(X)
logit_model = sm.Logit(y, X_full_const).fit(disp=0)

# The full statistical summary: coefficients, standard errors, p-values, pseudo R-squared
print(logit_model.summary())

In [ ]:
# Coefficients are log-odds and are not comparable across features with different scales.
# Converting them to "odds ratio per one standard deviation" makes the effects comparable.
feature_std = X.std()                                            # spread of each feature in the data
odds_ratio_per_sd = np.exp(logit_model.params[feature_names] * feature_std)

effects_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": logit_model.params[feature_names].to_numpy(),
    "p_value": logit_model.pvalues[feature_names].to_numpy(),
    "odds_ratio_per_1_sd": odds_ratio_per_sd.to_numpy()
}).sort_values("odds_ratio_per_1_sd", ascending=True)

# Readable labels for the chart
readable_labels = {
    "log_population": "Population (log10)",
    "income_10k": "Income per taxpayer (CHF 10k)",
    "external_competitor_count": "External competitors in postcode",
    "log_catchment_population": "Population within 5 km (log10)"
}
effects_df["label"] = effects_df["feature"].map(readable_labels)

# Bar chart of the odds ratios; a value above 1 raises the probability of a Migros, below 1 lowers it
fig_odds = px.bar(
    effects_df,
    x="odds_ratio_per_1_sd",
    y="label",
    orientation="h",
    text="odds_ratio_per_1_sd",
    color="odds_ratio_per_1_sd",
    color_continuous_scale="Oranges",
    title="What Predicts the Presence of a Migros? (odds ratio per 1 standard deviation)"
)

fig_odds.update_traces(texttemplate="%{text:.2f}x", textposition="outside", cliponaxis=False)

# A log x-axis is used because the population effect is an order of magnitude larger than the others
fig_odds.update_layout(
    xaxis=dict(title="Odds ratio (log scale)", type="log", gridcolor="#555555"),
    yaxis=dict(title="", gridcolor="#555555"),
    coloraxis_showscale=False,
    paper_bgcolor="#2B2B2B",
    plot_bgcolor="#3A3A3A",
    font=dict(color="white"),
    template="plotly_dark",
    title=dict(x=0.5, xanchor="center", font=dict(size=20, color="white")),
    height=480,
    width=1000,
    margin=dict(l=260, r=100, t=90, b=60)
)

# Reference line at 1 = "no effect"
fig_odds.add_vline(x=1, line_dash="dash", line_color="white", line_width=1)

fig_odds.show()
display(effects_df[["label", "coefficient", "odds_ratio_per_1_sd", "p_value"]].round(4))

### 2.2 From probability to coverage gap

The model now predicts, for every postcode, the probability that a postcode with that profile
contains a Migros. For the four candidates the actual value is **0 Migros stores**, so:

> **high predicted probability + no Migros = the clearest coverage gap**

This is the key difference from notebook 07: the ranking there rewarded high income, while the
model weights each factor by how strongly it actually separates Migros from non-Migros postcodes
across the whole country.

In [ ]:
# Apply the fitted model to every postcode
scoring_df["predicted_migros_probability"] = logit_model.predict(X_full_const)

# The gap index is only meaningful where no Migros exists yet, so it is set to NaN elsewhere
scoring_df["coverage_gap_index"] = np.where(
    scoring_df["migros_count"] == 0,                        # condition: white spot
    scoring_df["predicted_migros_probability"] * 100,       # value if true: expressed as 0-100
    np.nan                                                  # value if false: not applicable
)

# Pull out the four candidates and keep them in the notebook 07 rank order
candidates_df = scoring_df[scoring_df["postal_code"].isin(top4_postcodes)].copy()
candidates_df["notebook07_rank"] = candidates_df["postal_code"].map(dict(zip(top4_df["postal_code"], top4_df["rank"])))
candidates_df = candidates_df.sort_values("notebook07_rank")

display(
    candidates_df[["notebook07_rank", "postal_code", "locality_name", "primary_canton", "population",
                   "avg_income_per_taxpayer", "external_competitor_count", "catchment_population_5km",
                   "predicted_migros_probability"]].style.format({
        "population": "{:,.0f}",
        "avg_income_per_taxpayer": "CHF {:,.0f}",
        "catchment_population_5km": "{:,.0f}",
        "predicted_migros_probability": "{:.1%}"
    })
)

In [ ]:
# Visual comparison of the four candidates: model probability versus the notebook 07 score
candidates_plot_df = candidates_df.copy()

# Label each bar with locality and postcode so the chart is readable on its own
candidates_plot_df["label"] = candidates_plot_df["locality_name"] + " (" + candidates_plot_df["postal_code"] + ")"

# Probability is converted to percent for the axis
candidates_plot_df["probability_pct"] = candidates_plot_df["predicted_migros_probability"] * 100

# Sort ascending because Plotly draws horizontal bars from the bottom upwards
candidates_plot_df = candidates_plot_df.sort_values("probability_pct", ascending=True)

fig_probability = px.bar(
    candidates_plot_df,
    x="probability_pct",
    y="label",
    orientation="h",
    text="probability_pct",
    color="probability_pct",
    color_continuous_scale="Oranges",
    custom_data=["population", "avg_income_per_taxpayer", "external_competitor_count", "catchment_population_5km"],
    title="Modelled Probability That This Postcode Should Have a Migros"
)

fig_probability.update_traces(
    texttemplate="%{text:.1f}%",
    textposition="outside",
    cliponaxis=False,
    hovertemplate=(
        "<b>%{y}</b><br>"
        "Model probability: %{x:.1f}%<br>"
        "Population: %{customdata[0]:,.0f}<br>"
        "Average income: CHF %{customdata[1]:,.0f}<br>"
        "External competitors: %{customdata[2]:,.0f}<br>"
        "Population within 5 km: %{customdata[3]:,.0f}"
        "<extra></extra>"
    )
)

fig_probability.update_layout(
    xaxis=dict(title="Predicted probability of containing a Migros (%)", range=[0, 110], gridcolor="#555555"),
    yaxis=dict(title="", gridcolor="#555555"),
    coloraxis_showscale=False,
    paper_bgcolor="#2B2B2B",
    plot_bgcolor="#3A3A3A",
    font=dict(color="white"),
    template="plotly_dark",
    title=dict(x=0.5, xanchor="center", font=dict(size=21, color="white")),
    height=480,
    width=1050,
    margin=dict(l=210, r=100, t=90, b=60)
)

fig_probability.show()

### 2.3 Conclusion of Part A

Read the chart as: *"postcodes that look like this one have a Migros X% of the time - but this one
has none."*

- **Siebnen (8854)** stands far above the rest. It has the largest population of the four, three
  external competitors proving that supermarket demand exists, and still no Migros. The model
  essentially says a Migros "should" be there.
- **Kilchberg (8802)** scored first in notebook 07 because of its exceptional income, but the
  model gives income a *negative* coefficient once size is controlled for (wealthy commuter
  suburbs are typically served by neighbouring centres), and notebook 07 already showed two
  Migros plus three Denner stores within 2 km - a cannibalisation risk rather than a gap.
- **Wetzikon (8623)** and **Aarburg (4663)** sit in the middle: solid populations, but no
  competitor presence in the postcode and dense surrounding coverage.

**Recommendation: open the new store in Siebnen, postcode 8854 (canton Schwyz).**
The rest of the notebook answers *where inside Siebnen*.

## 3. Part B - Where exactly in Siebnen?

Postcode 8854 is roughly 4 km wide, so "8854" is not yet a site. The approach:

1. **Demand points** - split the 11,369 residents of 8854 over the sub-areas of the postcode using
   the official address shares, and add the surrounding postcodes as spill-over demand.
2. **Supply points** - every existing supermarket within 10 km, weighted by format size
   (a Coop or Migros pulls more than a small Denner).
3. **A 100 m grid** over the village, restricted to realistic parcels.
4. **A Huff gravity model** for each grid cell: how many customers would a new Migros at that
   exact point capture, given everything already competing for them?
5. **Accessibility**: real supermarkets sit on the main through-road, so cells far from the
   Zurcherstrasse / Glarnerstrasse axis are down-weighted.

In [ ]:
# Official Swiss locality directory (Amtliches Ortschaftenverzeichnis) - it is read directly from the
# ZIP archive, which avoids having to unpack it and keeps the data folder clean.
ortschaften_zip = data_folder / "ortschaftenverzeichnis_plz_4326_csv.zip"

with zipfile.ZipFile(ortschaften_zip) as archive:
    # The archive contains one CSV plus a folder entry, so the CSV is selected explicitly
    csv_name = [name for name in archive.namelist() if name.lower().endswith(".csv")][0]
    with archive.open(csv_name) as csv_file:
        localities_df = pd.read_csv(csv_file, sep=";")     # this file is semicolon-separated

# Keep the rows belonging to postcode 8854.
# One postcode can be split over several municipalities; each row gives that part's
# share of all addresses (Adressenanteil) plus its coordinates (E = longitude, N = latitude).
siebnen_parts = localities_df[localities_df["PLZ4"] == 8854].copy()

# "51.158 %" is text - strip the percent sign and convert to a 0-1 fraction
siebnen_parts["address_share"] = siebnen_parts["Adressenanteil"].astype(str).str.replace("%", "", regex=False).str.strip().astype(float) / 100

display(siebnen_parts[["Ortschaftsname", "Gemeindename", "address_share", "N", "E", "ZIP_ID"]])

In [ ]:
# Postcode 8854 covers two settlements: Siebnen itself (ZIP_ID 5088) and the village of
# Galgenen (ZIP_ID 5089). The population figure from notebook 06 is for the whole postcode,
# so it has to be split between them.
SIEBNEN_POPULATION_SHARE = 0.70      # assumption: Siebnen is the larger of the two settlements
                                     # (tunable - refine with BFS hectare-grid data if available)

SIEBNEN_ZIP_ID = 5088                # identifier of the Siebnen locality in the official file

# Total residents of postcode 8854, taken straight from the master table
siebnen_total_population = float(scoring_df.loc[scoring_df["postal_code"] == "8854", "population"].iloc[0])

demand_rows = []
for _, part in siebnen_parts.iterrows():
    if part["ZIP_ID"] == SIEBNEN_ZIP_ID:
        # Inside Siebnen: distribute the Siebnen share further by official address share
        weight = SIEBNEN_POPULATION_SHARE * part["address_share"]
    else:
        # The Galgenen village part gets the remaining share
        weight = (1 - SIEBNEN_POPULATION_SHARE) * part["address_share"]

    demand_rows.append({
        "name": f"{part['Ortschaftsname']} ({part['Gemeindename']})",
        "latitude": part["N"],
        "longitude": part["E"],
        "population": siebnen_total_population * weight,
        "is_local": part["ZIP_ID"] == SIEBNEN_ZIP_ID      # marks the cells that define the Siebnen village area
    })

# Geographic reference point for the whole Part B analysis: the centre of postcode 8854
siebnen_center_lat = float(scoring_df.loc[scoring_df["postal_code"] == "8854", "postcode_latitude"].iloc[0])
siebnen_center_lon = float(scoring_df.loc[scoring_df["postal_code"] == "8854", "postcode_longitude"].iloc[0])

# Surrounding postcodes also send customers to a store in Siebnen, so they are added as
# demand points at their own centre. 8 km is wide enough to include Lachen, Wangen, Tuggen,
# Schubelbach and Buttikon without dragging in the Rapperswil agglomeration.
neighbour_distance = haversine_km(siebnen_center_lat, siebnen_center_lon, scoring_df["postcode_latitude"], scoring_df["postcode_longitude"])

neighbours_df = scoring_df[(neighbour_distance < 8) & (scoring_df["postal_code"] != "8854")]

for _, neighbour in neighbours_df.iterrows():
    demand_rows.append({
        "name": f"{neighbour['locality_name']} ({neighbour['postal_code']})",
        "latitude": neighbour["postcode_latitude"],
        "longitude": neighbour["postcode_longitude"],
        "population": float(neighbour["population"]),
        "is_local": False
    })

demand_df = pd.DataFrame(demand_rows)

print("Demand points:", len(demand_df), "| total population represented:", f"{demand_df['population'].sum():,.0f}")
display(demand_df[demand_df["is_local"]])

In [ ]:
# Existing stores are the competition in the gravity model.
stores_df = pd.read_csv(data_folder / "switzerland_supermarkets_clean.csv")

# Distance of every Swiss store from the centre of Siebnen
stores_df["distance_from_siebnen_km"] = haversine_km(siebnen_center_lat, siebnen_center_lon, stores_df["latitude"], stores_df["longitude"])

# Keep everything within 10 km: stores further away barely influence a choice made in Siebnen,
# but the ones in Lachen, Buttikon and Reichenburg definitely do
local_stores_df = stores_df[stores_df["distance_from_siebnen_km"] <= 10].copy()

# Attractiveness weights reflect typical sales area per format: a full-range Migros/Coop
# supermarket draws more trips than a discounter, and a small Denner satellite draws the least.
attractiveness_weights = {"Migros": 1.0, "Coop": 1.0, "Aldi": 0.8, "Lidl": 0.8, "Denner": 0.5}
local_stores_df["attractiveness"] = local_stores_df["store_group"].map(attractiveness_weights).fillna(0.6)

print("Competing stores within 10 km:", len(local_stores_df))

# The stores physically inside Siebnen are the ones the new store would compete with directly
display(
    local_stores_df[local_stores_df["distance_from_siebnen_km"] <= 2][
        ["name", "store_group", "street", "house_number", "latitude", "longitude", "distance_from_siebnen_km", "attractiveness"]
    ].sort_values("distance_from_siebnen_km").style.format({"distance_from_siebnen_km": "{:.2f} km"})
)

In [ ]:
# Build a regular grid of candidate points over the Siebnen village area.
local_demand_df = demand_df[demand_df["is_local"]]     # the three sub-areas that make up Siebnen

# Grid spacing: 0.0007 degrees of latitude is about 78 m, 0.0010 degrees of longitude about 76 m
# at this latitude - fine enough to distinguish individual street blocks.
GRID_LAT_STEP = 0.0007
GRID_LON_STEP = 0.0010

# The box extends about 1 km beyond the outermost sub-area so nothing is cut off
grid_latitudes = np.arange(local_demand_df["latitude"].min() - 0.010, local_demand_df["latitude"].max() + 0.010, GRID_LAT_STEP)
grid_longitudes = np.arange(local_demand_df["longitude"].min() - 0.016, local_demand_df["longitude"].max() + 0.016, GRID_LON_STEP)

# Cartesian product of both axes = every grid point
grid_points = np.array([(lat, lon) for lat in grid_latitudes for lon in grid_longitudes])

print("Raw grid cells:", len(grid_points))

In [ ]:
# The raw grid also covers fields, the Linth plain and the slopes above the village,
# so three filters keep only plausible retail parcels.

# Filter 1: within 900 m of one of the Siebnen sub-area centres -> inside the built-up village
distance_to_village = np.min([haversine_km(grid_points[:, 0], grid_points[:, 1], row["latitude"], row["longitude"])
                              for _, row in local_demand_df.iterrows()], axis=0)

# Filter 2: at least 250 m away from an existing supermarket - a new store cannot be built
# on top of a competitor, and sites that close would share the same parcel anyway
distance_to_existing_store = np.min([haversine_km(grid_points[:, 0], grid_points[:, 1], row["latitude"], row["longitude"])
                                     for _, row in local_stores_df.iterrows()], axis=0)

# Filter 3 needs the main retail axis first. All four supermarkets in Siebnen sit on the
# Zurcherstrasse / Glarnerstrasse through-road, so the line connecting them is a good proxy
# for the main road, without needing an external road dataset.
retail_axis = local_stores_df[local_stores_df["distance_from_siebnen_km"] <= 2].sort_values("longitude")[["latitude", "longitude"]].to_numpy()

def distance_to_segment_km(points, segment_start, segment_end):
    """Shortest distance from each point to a straight segment, in km (local flat-earth approximation)."""
    km_per_degree = 111.32                                   # length of one degree of latitude in km
    lon_scale = np.cos(np.radians(points[:, 0].mean()))      # degrees of longitude shrink with latitude

    # Convert everything to local x/y kilometres relative to the segment start
    px = (points[:, 1] - segment_start[1]) * km_per_degree * lon_scale
    py = (points[:, 0] - segment_start[0]) * km_per_degree
    bx = (segment_end[1] - segment_start[1]) * km_per_degree * lon_scale
    by = (segment_end[0] - segment_start[0]) * km_per_degree

    # Project each point onto the segment and clamp to [0, 1] so the projection stays on the segment
    t = np.clip((px * bx + py * by) / (bx * bx + by * by), 0, 1)

    # Euclidean distance between the point and its projection
    return np.hypot(px - t * bx, py - t * by)

# Distance to the closest piece of the multi-segment axis
distance_to_axis = np.min([distance_to_segment_km(grid_points, retail_axis[i], retail_axis[i + 1])
                           for i in range(len(retail_axis) - 1)], axis=0)

# Apply all three filters at once
keep_mask = (distance_to_village <= 0.9) & (distance_to_existing_store >= 0.25) & (distance_to_axis <= 1.0)

candidate_sites = grid_points[keep_mask]
candidate_axis_distance = distance_to_axis[keep_mask]
candidate_store_distance = distance_to_existing_store[keep_mask]

print("Candidate cells after filtering:", len(candidate_sites))

### 3.1 The Huff model

For a shopper in area *i*, the probability of choosing store *j* is its attractiveness divided by
the distance penalty, relative to **all** stores available:

$$P_{ij} = \frac{A_j\,/\,d_{ij}^{\beta}}{\sum_k A_k\,/\,d_{ik}^{\beta}}$$

with $\beta = 2$ (the standard grocery value - a store twice as far is four times less likely to
be chosen). Multiplying $P_{ij}$ by the population of area *i* and summing over all areas gives
the **expected number of customers captured** by a new store at that point.

In [ ]:
DISTANCE_DECAY = 2.0        # beta: how strongly shoppers dislike travelling (2 is standard for groceries)
DISTANCE_FLOOR_KM = 0.2     # minimum distance, so a store sitting exactly on a demand point does not divide by zero
NEW_STORE_ATTRACTIVENESS = 1.0   # a full-format Migros supermarket, same weight as an existing Migros/Coop

# Step 1: for every demand point, how much pull do the EXISTING stores already exert?
# This denominator does not depend on the candidate site, so it is computed once.
existing_pull = np.array([
    np.sum(local_stores_df["attractiveness"].to_numpy() /
           np.maximum(haversine_km(row["latitude"], row["longitude"],
                                   local_stores_df["latitude"].to_numpy(),
                                   local_stores_df["longitude"].to_numpy()),
                      DISTANCE_FLOOR_KM) ** DISTANCE_DECAY)
    for _, row in demand_df.iterrows()
])

# Step 2: accumulate the captured population over all demand points for every candidate cell
captured_population = np.zeros(len(candidate_sites))

for i, demand_point in demand_df.iterrows():
    # Distance from every candidate cell to this demand point
    distance = np.maximum(
        haversine_km(candidate_sites[:, 0], candidate_sites[:, 1], demand_point["latitude"], demand_point["longitude"]),
        DISTANCE_FLOOR_KM
    )

    # Pull of the hypothetical new store on this demand point
    new_store_pull = NEW_STORE_ATTRACTIVENESS / distance ** DISTANCE_DECAY

    # Huff share = own pull / (own pull + everything already there), multiplied by the residents
    captured_population += demand_point["population"] * new_store_pull / (new_store_pull + existing_pull[i])

print("Expected captured customers - best cell:", f"{captured_population.max():,.0f}")
print("Expected captured customers - worst cell:", f"{captured_population.min():,.0f}")

In [ ]:
# Accessibility: a parcel 800 m off the main road is far less viable than one on it,
# even if the raw gravity score is similar (no passing traffic, no bus stop, worse delivery access).
# An exponential decay with a 350 m scale halves the score at roughly 240 m off the axis.
ACCESS_DECAY_KM = 0.35
accessibility = np.exp(-candidate_axis_distance / ACCESS_DECAY_KM)

# Final site score = expected customers captured, discounted by accessibility
site_scores_df = pd.DataFrame({
    "latitude": candidate_sites[:, 0],
    "longitude": candidate_sites[:, 1],
    "captured_population": captured_population,
    "distance_to_axis_km": candidate_axis_distance,
    "distance_to_nearest_store_km": candidate_store_distance,
    "accessibility": accessibility
})
site_scores_df["site_score"] = site_scores_df["captured_population"] * site_scores_df["accessibility"]

# Rank all cells from best to worst
site_scores_df = site_scores_df.sort_values("site_score", ascending=False).reset_index(drop=True)

display(site_scores_df.head(5).round(3))

In [ ]:
# The best cells are all neighbours of each other, so picking the top 3 rows would return
# three points on the same parcel. A greedy selection enforces a minimum separation instead.
MINIMUM_SITE_SEPARATION_KM = 0.45     # roughly one block apart, so the options are genuinely different

selected_sites = []
for _, cell in site_scores_df.iterrows():
    # Accept the cell only if it is far enough from every site already chosen
    if all(haversine_km(cell["latitude"], cell["longitude"], chosen["latitude"], chosen["longitude"]) > MINIMUM_SITE_SEPARATION_KM
           for chosen in selected_sites):
        selected_sites.append(cell)
    if len(selected_sites) == 3:      # three options are enough to present to a decision maker
        break

suggested_sites_df = pd.DataFrame(selected_sites).reset_index(drop=True)
suggested_sites_df["site_rank"] = suggested_sites_df.index + 1

# Describe each site by its nearest known landmark (an existing store with a street address),
# because coordinates alone are hard to picture
nearest_store_names = []
nearest_migros_distances = []
migros_stores = local_stores_df[local_stores_df["store_group"] == "Migros"]

for _, site in suggested_sites_df.iterrows():
    distances = haversine_km(site["latitude"], site["longitude"], local_stores_df["latitude"].to_numpy(), local_stores_df["longitude"].to_numpy())
    nearest = local_stores_df.iloc[int(np.argmin(distances))]
    nearest_store_names.append(f"{nearest['name']}, {nearest['street']} {nearest['house_number']} ({np.min(distances) * 1000:.0f} m)")

    # Distance to the closest existing Migros = cannibalisation check
    nearest_migros_distances.append(haversine_km(site["latitude"], site["longitude"], migros_stores["latitude"].to_numpy(), migros_stores["longitude"].to_numpy()).min())

suggested_sites_df["nearest_existing_store"] = nearest_store_names
suggested_sites_df["distance_to_nearest_migros_km"] = nearest_migros_distances

display(
    suggested_sites_df[["site_rank", "latitude", "longitude", "captured_population", "site_score",
                        "distance_to_axis_km", "nearest_existing_store", "distance_to_nearest_migros_km"]].style.format({
        "latitude": "{:.5f}",
        "longitude": "{:.5f}",
        "captured_population": "{:,.0f}",
        "site_score": "{:,.0f}",
        "distance_to_axis_km": "{:.2f} km",
        "distance_to_nearest_migros_km": "{:.2f} km"
    })
)

### 3.2 The site map

The map shows three layers: the modelled attractiveness surface over Siebnen, the supermarkets
already operating there, and the three recommended sites.

In [ ]:
# Build the map figure layer by layer
fig_site_map = go.Figure()

# Layer 1: the score surface as a heat layer. Weighting every grid cell by its own score makes
# the strong corridors glow and the weak edges fade.
fig_site_map.add_trace(
    DensityMapTrace(
        lat=site_scores_df["latitude"],
        lon=site_scores_df["longitude"],
        z=site_scores_df["site_score"],
        radius=22,                       # smoothing radius in pixels
        colorscale="YlOrRd",
        opacity=0.55,
        colorbar=dict(title="Site score<br>(customers x access)"),
        hoverinfo="skip",                # the surface is background; details come from the markers
        name="Attractiveness surface"
    )
)

# Layer 2: demand points inside Siebnen, sized by the residents they represent
local_demand_plot = demand_df[demand_df["is_local"]]
fig_site_map.add_trace(
    ScatterMapTrace(
        lat=local_demand_plot["latitude"],
        lon=local_demand_plot["longitude"],
        mode="markers",
        marker=dict(size=18, color="#1F77B4", opacity=0.75),
        text=local_demand_plot["name"],
        customdata=local_demand_plot["population"],
        hovertemplate="<b>%{text}</b><br>Residents: %{customdata:,.0f}<extra></extra>",
        name="Demand centres (8854)"
    )
)

# Layer 3: the competing stores, one trace per brand so the legend can toggle them
brand_colors = {"Migros": "#F58220", "Coop": "#E30613", "Denner": "#19A831", "Aldi": "#120DA9", "Lidl": "#8B0E70"}

for brand, brand_stores in local_stores_df[local_stores_df["distance_from_siebnen_km"] <= 4].groupby("store_group"):
    fig_site_map.add_trace(
        ScatterMapTrace(
            lat=brand_stores["latitude"],
            lon=brand_stores["longitude"],
            mode="markers",
            marker=dict(size=13, color=brand_colors.get(brand, "#777777")),
            # fillna keeps the hover text readable where OSM has no address
            text=brand + " - " + brand_stores["street"].fillna("address unknown").astype(str) + " " + brand_stores["house_number"].fillna("").astype(str),
            hovertemplate="<b>%{text}</b><extra></extra>",
            name=brand
        )
    )

# Layer 4: the recommendations, drawn last so they sit on top of everything else
fig_site_map.add_trace(
    ScatterMapTrace(
        lat=suggested_sites_df["latitude"],
        lon=suggested_sites_df["longitude"],
        mode="markers+text",
        marker=dict(size=26, color="#00E5FF", opacity=0.95),
        text=["#" + str(rank) for rank in suggested_sites_df["site_rank"]],
        textfont=dict(size=15, color="#04101E"),
        customdata=suggested_sites_df[["captured_population", "nearest_existing_store", "distance_to_nearest_migros_km"]].to_numpy(),
        hovertemplate=(
            "<b>Suggested site %{text}</b><br>"
            "Coordinates: %{lat:.5f}, %{lon:.5f}<br>"
            "Expected customers captured: %{customdata[0]:,.0f}<br>"
            "Nearest existing store: %{customdata[1]}<br>"
            "Nearest Migros: %{customdata[2]:.2f} km"
            "<extra></extra>"
        ),
        name="Recommended sites"
    )
)

# Centre the basemap on the village and zoom in far enough to see individual streets
apply_map_layout(fig_site_map, siebnen_center_lat, siebnen_center_lon, 13.6)

fig_site_map.update_layout(
    title=dict(text="Where to Open the New Migros in Siebnen (8854)", x=0.5, xanchor="center", font=dict(size=22)),
    height=780,
    width=1150,
    margin=dict(l=20, r=20, t=70, b=20),
    legend=dict(bgcolor="rgba(255,255,255,0.85)", bordercolor="#333333", borderwidth=1, x=0.01, y=0.99)
)

fig_site_map.show()

In [ ]:
# A second, wider map puts the recommendation in its regional context: it shows why Siebnen
# is a gap at all - the nearest Migros stores are in Lachen and to the east.
fig_context_map = go.Figure()

# Existing stores within 10 km, coloured by brand
for brand, brand_stores in local_stores_df.groupby("store_group"):
    fig_context_map.add_trace(
        ScatterMapTrace(
            lat=brand_stores["latitude"],
            lon=brand_stores["longitude"],
            mode="markers",
            marker=dict(size=11, color=brand_colors.get(brand, "#777777")),
            text=brand + " - " + brand_stores["city"].fillna("").astype(str),
            hovertemplate="<b>%{text}</b><extra></extra>",
            name=brand
        )
    )

# Surrounding demand, sized by population, to show where the catchment actually lives
fig_context_map.add_trace(
    ScatterMapTrace(
        lat=demand_df["latitude"],
        lon=demand_df["longitude"],
        mode="markers",
        # Square-root scaling keeps a 19,000-resident town from swamping a 1,000-resident village
        marker=dict(size=np.sqrt(demand_df["population"]) / 4 + 6, color="#1F77B4", opacity=0.45),
        text=demand_df["name"],
        customdata=demand_df["population"],
        hovertemplate="<b>%{text}</b><br>Residents: %{customdata:,.0f}<extra></extra>",
        name="Demand (population)"
    )
)

# The single best site
best_site = suggested_sites_df.iloc[0]
fig_context_map.add_trace(
    ScatterMapTrace(
        lat=[best_site["latitude"]],
        lon=[best_site["longitude"]],
        mode="markers+text",
        marker=dict(size=24, color="#00E5FF"),
        text=["NEW"],
        textfont=dict(size=13, color="#04101E"),
        hovertemplate="<b>Recommended new Migros</b><br>%{lat:.5f}, %{lon:.5f}<extra></extra>",
        name="Recommended new Migros"
    )
)

# Zoomed out so Lachen, Wangen, Tuggen and Buttikon are all visible
apply_map_layout(fig_context_map, siebnen_center_lat, siebnen_center_lon, 11.2)

fig_context_map.update_layout(
    title=dict(text="Regional Context: Siebnen and the Existing Supermarket Network", x=0.5, xanchor="center", font=dict(size=22)),
    height=720,
    width=1150,
    margin=dict(l=20, r=20, t=70, b=20),
    legend=dict(bgcolor="rgba(255,255,255,0.85)", bordercolor="#333333", borderwidth=1, x=0.01, y=0.99)
)

fig_context_map.show()

In [ ]:
# Print a short, quotable summary of the final recommendation
best_site = suggested_sites_df.iloc[0]

print("FINAL RECOMMENDATION")
print("=" * 60)
print("Postcode          : 8854 Siebnen (canton Schwyz)")
print("Model probability : "
      f"{scoring_df.loc[scoring_df['postal_code'] == '8854', 'predicted_migros_probability'].iloc[0]:.1%}"
      " that a postcode with this profile has a Migros - it has none")
print("Postcode residents:", f"{siebnen_total_population:,.0f}")
print()
print("Recommended site  :", f"{best_site['latitude']:.5f}, {best_site['longitude']:.5f}")
print("Landmark          :", best_site["nearest_existing_store"])
print("Expected customers:", f"{best_site['captured_population']:,.0f} (Huff model, beta = {DISTANCE_DECAY})")
print("Nearest Migros    :", f"{best_site['distance_to_nearest_migros_km']:.2f} km (low cannibalisation risk)")
print()
print("Alternatives:")
for _, site in suggested_sites_df.iloc[1:].iterrows():
    print(f"  #{int(site['site_rank'])}: {site['latitude']:.5f}, {site['longitude']:.5f} - "
          f"{site['captured_population']:,.0f} customers - near {site['nearest_existing_store']}")

## 4. Conclusions

**Where to open the store: Siebnen, postcode 8854.**
The logistic regression, trained on all Swiss postcodes, gives Siebnen by far the highest
probability of "should have a Migros" among the four candidates, while it actually has none.
Kilchberg, which topped the hand-weighted ranking of notebook 07, drops back because the model
shows income matters much less than population and proven retail demand once everything is
weighed against real data - and because it is already surrounded by Migros Group stores.

**Where inside Siebnen:** on the Zurcherstrasse / Glarnerstrasse retail spine, in the gap between
the existing store clusters. All three suggested sites sit on this axis; the top-ranked one
captures the most demand while staying at least 250 m from any competitor and more than 3 km from
the nearest Migros.

### Limitations, honestly stated

1. **The 70/30 population split** between Siebnen and Galgenen inside postcode 8854 is an
   assumption. BFS hectare-grid population data would replace it with measured values.
2. **No land-use, zoning or parcel-availability data.** The model says where demand is; it cannot
   say whether a plot is for sale or zoned for retail.
3. **The "main road" is approximated** by the line through the existing supermarkets. A real road
   network (OpenStreetMap) plus traffic counts would be more precise, and driving times would be
   better than straight-line distances in an alpine valley.
4. **The Huff parameters** (attractiveness weights, beta = 2) are literature defaults, not
   calibrated on Swiss trip data.
5. **The logistic regression describes where Migros stores are today**, which reflects past
   expansion decisions. It identifies gaps relative to that pattern, not profitability - a real
   business case would need revenue, rent and construction costs.

In [ ]:
# Save every figure of this notebook as a standalone interactive HTML file,
# using the same pattern as the end of notebook 07
figure_variables = {
    "10_logit_odds_ratios": "fig_odds",
    "11_candidate_probabilities": "fig_probability",
    "12_siebnen_site_map": "fig_site_map",
    "13_siebnen_regional_context": "fig_context_map"
}

# Only save figures whose cells have actually been executed
figures_to_save = {name: globals()[variable] for name, variable in figure_variables.items() if variable in globals()}

for name, figure in figures_to_save.items():
    figure.write_html(plot_folder / f"{name}.html", include_plotlyjs=True, full_html=True)

print(f"Saved {len(figures_to_save)} interactive HTML figures in '{plot_folder}'.")

# Also export the suggested sites so they can be opened in Google Maps or QGIS
suggested_sites_df.to_csv(data_folder / "siebnen_suggested_sites.csv", index=False)
print("Saved suggested site coordinates to data/siebnen_suggested_sites.csv")